In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd

In [10]:
CBE_APP_ID = "com.combanketh.mobilebanking"
BOA_APP_ID = "com.boa.boaMobileBanking"
DASHEN_APP_ID = "com.dashen.dashensuperapp"

In [11]:
def scrape_bank_reviews(app_id, bank_name, count=500):
    """
    Scrape reviews from Google Play Store.
    """

    result, continuation_token = reviews(
        app_id,
        lang="en",
        country="et",
        sort=Sort.NEWEST,
        count=count
    )

    review_data = []

    for review in result:
        review_data.append({
            "review_id": review["reviewId"],
            "review": review["content"],
            "rating": review["score"],
            "date": review["at"].strftime("%Y-%m-%d"),
            "bank": bank_name,
            "source": "Google Play"
        })

    df = pd.DataFrame(review_data)

    return df

In [12]:
# Commercial Bank of Ethiopia
df_cbe = scrape_bank_reviews(
    CBE_APP_ID,
    "Commercial Bank of Ethiopia",
    count=500
)

# Bank of Abyssinia
df_boa = scrape_bank_reviews(
    BOA_APP_ID,
    "Bank of Abyssinia",
    count=500
)

# Dashen Bank
df_dashen = scrape_bank_reviews(
    DASHEN_APP_ID,
    "Dashen Bank",
    count=500
)

In [13]:
df_reviews = pd.concat(
    [df_cbe, df_boa, df_dashen],
    ignore_index=True
)

print("Combined dataset shape:", df_reviews.shape)

df_reviews.head()

Combined dataset shape: (1500, 6)


,review_id,review,rating,date,bank,source
0,dd4ae5b5-f5a4-42e9-a526-5fe9387dd7a4,good,5,2026-05-12,Commercial Bank of Ethiopia,Google Play
1,e9233eb9-c338-4c80-8f21-26cd31acd65f,Good to use,5,2026-05-12,Commercial Bank of Ethiopia,Google Play
2,6b0d612d-f9c1-4cac-bb37-b945c97a2e9a,cbe,1,2026-05-12,Commercial Bank of Ethiopia,Google Play
3,68ff6a46-3659-47f0-a20f-fba5755a3f67,Cbe,4,2026-05-11,Commercial Bank of Ethiopia,Google Play
4,95f3c528-c059-4a5a-8973-46182fe5cb22,best and secured,5,2026-05-11,Commercial Bank of Ethiopia,Google Play


In [14]:
# Initial shape
print("Initial dataset shape:", df_reviews.shape)

# Missing values
print("\nMissing values before cleaning:")
print(df_reviews.isnull().sum())

# Store initial rows
initial_rows = len(df_reviews)

# Drop missing review or rating
df_reviews = df_reviews.dropna(
    subset=["review", "rating"]
)

missing_removed = initial_rows - len(df_reviews)

print(f"\nRows removed due to missing values: {missing_removed}")

# Remove duplicates using review_id
before_duplicates = len(df_reviews)

df_reviews = df_reviews.drop_duplicates(
    subset=["review_id"]
)

duplicates_removed = before_duplicates - len(df_reviews)

print(f"Duplicate rows removed: {duplicates_removed}")

# Normalize date
df_reviews["date"] = pd.to_datetime(
    df_reviews["date"]
).dt.strftime("%Y-%m-%d")

# Final shape
print("\nFinal dataset shape:", df_reviews.shape)

# Missing percentage
missing_percentage = (
    df_reviews.isnull().sum().sum()
    / (df_reviews.shape[0] * df_reviews.shape[1])
) * 100

print(f"\nMissing data percentage: {missing_percentage:.2f}%")

Initial dataset shape: (1500, 6)

Missing values before cleaning:
review_id    0
review       0
rating       0
date         0
bank         0
source       0
dtype: int64

Rows removed due to missing values: 0
Duplicate rows removed: 0

Final dataset shape: (1500, 6)

Missing data percentage: 0.00%


In [15]:
# Reviews per bank
print(df_reviews["bank"].value_counts())

# Total reviews
print("\nTotal reviews collected:", len(df_reviews))

bank
Commercial Bank of Ethiopia    500
Bank of Abyssinia              500
Dashen Bank                    500
Name: count, dtype: int64

Total reviews collected: 1500


In [16]:
final_df = df_reviews[
    ["review", "rating", "date", "bank", "source"]
]

final_df.to_csv(
    "../data/raw/bank_reviews_cleaned.csv",
    index=False
)

print("Final cleaned dataset saved successfully.")


Final cleaned dataset saved successfully.
